# Evaluate a trained run

Scores `runs/<RUN>/weights/best.pt` on Drive at several input sizes and, optionally,
saves per-image predictions as one zip for the failure analysis on the Mac. Results go
to `runs/<RUN>/eval/` on Drive.

- Pothole metrics come from the whole split, road metrics from `<split>_real_road.txt`.
- `SPLIT = "test"` needs `FINAL = True`: test is opened once, for the chosen model.

1. Runtime → Change runtime type → T4 GPU.
2. Set the options below, then Runtime → Run all (allow Drive access).

In [ ]:
RUN = "baseline"  # run folder under DRIVE_DIR/runs
DATASET = "merged-v1"  # zip name in DRIVE_DIR/data, without .zip
SPLIT = "val"  # val | test (test needs FINAL = True)
SIZES = "640 480 320"  # input sizes to validate at
PREDS_IMGSZ = 640  # save predictions at this size for failures.py; 0 = don't
FINAL = False
DRIVE_DIR = "/content/drive/MyDrive/road-pothole-seg"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Get the latest code from GitHub (code reaches Colab only through git).
import os

if os.path.isdir("/content/road-pothole-seg"):
    !git -C /content/road-pothole-seg pull --ff-only
else:
    !git clone https://github.com/ngtdat9/road-pothole-seg.git /content/road-pothole-seg
%cd /content/road-pothole-seg
!git log --oneline -1
!pip install -q -e ".[train]"

In [ ]:
# Copy the dataset to the local disk: training from the Drive mount is very slow.
if not os.path.isdir(f"/content/{DATASET}"):
    !cp "{DRIVE_DIR}/data/{DATASET}.zip" /content/
    !unzip -q "/content/{DATASET}.zip" -d /content/
!wc -l /content/{DATASET}/*.txt

In [ ]:
weights = f"{DRIVE_DIR}/runs/{RUN}/weights/best.pt"
out = f"{DRIVE_DIR}/runs/{RUN}/eval"
args = f"--weights '{weights}' --data-root /content/{DATASET} --out '{out}'"
args += f" --split {SPLIT} --sizes {SIZES} --preds-imgsz {PREDS_IMGSZ}"
if FINAL:
    args += " --final"
!python -m training.scripts.evaluate {args}

In [ ]:
# Make sure the predictions zip is fully written to Drive before the session ends.
drive.flush_and_unmount()

Done: tell Claude on the Mac, which reads `eval_<split>.json` and runs the failure analysis.